In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# ============================================
# Step 2: Load and Prepare Dataset
# ============================================

data = fetch_california_housing(as_frame=True)

df = pd.concat(
    [data.data, data.target.rename("HousePrice")],
    axis=1
)

print(df.head())

# Separate Features and Target

X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]

# ============================================
# Step 3: Feature Scaling
# ============================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ============================================
# Step 4: Train-Test Split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

# ============================================
# Step 5: Detect Overfitting
# ============================================

tree = DecisionTreeRegressor(random_state=42)

tree.fit(X_train, y_train)

train_pred = tree.predict(X_train)
test_pred = tree.predict(X_test)

train_rmse = np.sqrt(mean_squared_error(
    y_train,
    train_pred
))

test_rmse = np.sqrt(mean_squared_error(
    y_test,
    test_pred
))

print("Training RMSE :", train_rmse)
print("Testing RMSE  :", test_rmse)

# ============================================
# Step 6: Cross Validation
# ============================================

cv_scores = cross_val_score(
    tree,
    X_scaled,
    y,
    scoring="neg_root_mean_squared_error",
    cv=5
)

cv_rmse = -cv_scores.mean()

print("Cross Validation RMSE :", cv_rmse)

# ============================================
# Step 7: Hyperparameter Tuning
# ============================================

param_grid = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [2, 5, 10]
}

grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)

# ============================================
# Step 8: Evaluate Optimized Model
# ============================================

best_tree = grid.best_estimator_

y_pred = best_tree.predict(X_test)

rmse = np.sqrt(mean_squared_error(
    y_test,
    y_pred
))

r2 = r2_score(
    y_test,
    y_pred
)

print("\nOptimized Model Performance")
print("---------------------------")
print("RMSE :", rmse)
print("R²   :", r2)

# ============================================
# Optional: Compare Models
# ============================================

linear = LinearRegression()
linear.fit(X_train, y_train)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

models = {
    "Linear Regression": linear,
    "Ridge Regression": ridge,
    "Optimized Decision Tree": best_tree
}

print("\nModel Comparison")
print("----------------")

for name, model in models.items():
    pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(
        y_test,
        pred
    ))

    r2 = r2_score(
        y_test,
        pred
    )

    print(f"{name}")
    print(f"RMSE = {rmse:.4f}")
    print(f"R²   = {r2:.4f}")
    print("-" * 30)

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  HousePrice  
0    -122.23       4.526  
1    -122.22       3.585  
2    -122.24       3.521  
3    -122.25       3.413  
4    -122.25       3.422  
Training RMSE : 3.218325866275131e-16
Testing RMSE  : 0.7030445773467542
Cross Validation RMSE : 0.8957031908951016
Best Parameters:
{'max_depth': 10, 'min_samples_split': 10}

Optimized Model Performance
---------------------------
RMSE : 0.6454300828015771
R²   : 0.6820992539714815

Model Comparison
----------------
Linear Regression
RMSE = 0.7456
R²   = 0.5758
---------------